In [ ]:
import pandas as pd
data='/kaggle/input/hmisogyny/final_labels.csv'

In [ ]:
df = pd.read_csv(data)

In [ ]:
df = df.rename(columns={"level_3": "label"})

In [ ]:
df['label'] = df['label'].apply(lambda x: 1 if x == "Misogynistic" else 0)

In [ ]:
df_train = df[df['split']=='train']
df_test = df[df['split']=='test']

In [ ]:
df_train_itc=df_train[df_train['strength']=='Nature of the abuse is Implicit']
df_train_etc=df_train[df_train['strength']=='Nature of the abuse is Explicit']

In [ ]:
def get_qc_examples(df):
    """Creates examples for the training and dev sets."""
    text_and_labels = list(zip(df['body'], df['label']))
    return text_and_labels[1:]

In [ ]:
train_data = get_qc_examples(df_train)
test_data = get_qc_examples(df_test)

In [ ]:
len(train_data)

In [ ]:
len(test_data)

In [ ]:
X=[_ for _,label in train_data]
y=[label for _,label in train_data ]
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.7, random_state=42, stratify=y)

In [ ]:
labelled_examples=[]
unlabeled_examples=[]
for i in range(len(X_train)):
    labelled_examples.append((X_train[i],y_train[i]))
for i in range(len(X_test)):
    unlabeled_examples.append((X_test[i],y_test[i]))

In [ ]:
print(len(labelled_examples),len(unlabeled_examples))

In [ ]:
from collections import Counter
train_counts = Counter(label for _, label in unlabeled_examples)
print("Labeled Examples Count:", train_counts)
# Count labels in test examples
test_counts = Counter(label for _, label in test_data)
print("Test Examples Count:", test_counts)

In [ ]:
import torch
import io
import torch.nn.functional as F
import torch.nn as nn
import random
import numpy as np
import time
import math
import pandas as pd
import datetime
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

In [ ]:
import transformers
from transformers import *

In [ ]:
print(transformers.__version__)

In [ ]:
seed_val = 123
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(seed_val)

In [ ]:
max_seq_length = 256
batch_size = 16
noise_size = 100
out_dropout_rate = 0.2
apply_balance = True
learning_rate_discriminator = 3e-5
learning_rate_generator = 6e-4
# learning_rate_classifer = 5e-6
epsilon = 1e-8
num_train_epochs = 20
multi_gpu = True
apply_scheduler = False
warmup_proportion = 0.01
print_each_n_step = 100
num_labels=2
model_name="google-bert/bert-base-uncased"
label_list = [0,1]

In [ ]:
if torch.cuda.is_available():    
    # Tell PyTorch to use the GPU.    
    device = torch.device("cuda")
    print('There are %d GPU(s) available.' % torch.cuda.device_count())
    print('We will use the GPU:', torch.cuda.get_device_name(0))
# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [ ]:
transformer = AutoModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(transformer,tokenizer)

In [ ]:
def generate_data_loader(input_examples, label_masks, do_shuffle = False, balance_label_examples = False):
  '''
  Generate a Dataloader given the input examples, eventually masked if they are 
  to be considered NOT labeled.
  '''
  examples = []

  # Count the percentage of labeled examples  
  num_labeled_examples = 0
  for label_mask in label_masks:
    if label_mask: 
      num_labeled_examples += 1
  label_mask_rate = num_labeled_examples/len(input_examples)

  # if required it applies the balance
  for index, ex in enumerate(input_examples): 
    if label_mask_rate == 1 or not balance_label_examples:
      examples.append((ex, label_masks[index]))
    else:
      # IT SIMULATE A LABELED EXAMPLE
      if label_masks[index]:
        balance = int(1/label_mask_rate)
        balance = int(math.log(balance,2))
        if balance < 1:
          balance = 1
        for b in range(0, int(balance)):
          examples.append((ex, label_masks[index]))
      else:
        examples.append((ex, label_masks[index]))
  
  #-----------------------------------------------
  # Generate input examples to the Transformer
  #-----------------------------------------------
  input_ids = []
  input_mask_array = []
  label_mask_array = []
  label_id_array = []

  # Tokenization 
  for (text, label_mask) in examples:
    # print(type(text[0]))
    encoded_sent = tokenizer.encode(str(text[0]), add_special_tokens=True, max_length=max_seq_length, padding="max_length", truncation=True)
    input_ids.append(encoded_sent)
    label_id_array.append(int(text[1]))
    label_mask_array.append(int(label_mask))
  
  # Attention to token (to ignore padded input wordpieces)
  for sent in input_ids:
    att_mask = [int(token_id > 0) for token_id in sent]                          
    input_mask_array.append(att_mask)
  # Convertion to Tensor
  input_ids = torch.tensor(input_ids) 
  input_mask_array = torch.tensor(input_mask_array)
  label_id_array = torch.tensor(label_id_array, dtype=torch.long)
  label_mask_array = torch.tensor(label_mask_array)

  # Building the TensorDataset
  dataset = TensorDataset(input_ids, input_mask_array, label_id_array, label_mask_array)

  if do_shuffle:
    sampler = RandomSampler
  else:
    sampler = SequentialSampler

  # Building the DataLoader
  return DataLoader(
              dataset,  # The training samples.
              sampler = sampler(dataset), 
              batch_size = batch_size) # Trains with this batch size.

def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [ ]:
# label_map = {}
# for (i, label) in enumerate(label_list):
#   label_map[label] = i
#------------------------------
#   Load the train dataset
#------------------------------
# unlabeled_examples=train_data[0:len(train_data)//2]
# labelled_examples=train_data[len(train_data)//2:]
train_examples = labelled_examples
#The labeled (train) dataset is assigned with a mask set to True
train_label_masks = np.ones(len(labelled_examples), dtype=int)
#If unlabel examples are available
if unlabeled_examples:
  train_examples = train_examples + unlabeled_examples
  #The unlabeled (train) dataset is assigned with a mask set to False
  tmp_masks = np.zeros(len(unlabeled_examples), dtype=int)
  train_label_masks = np.concatenate([train_label_masks,tmp_masks])
# print(train_examples)
train_dataloader = generate_data_loader(train_examples, train_label_masks, do_shuffle = True, balance_label_examples = apply_balance)
#------------------------------
#   Load the test dataset
#------------------------------
#The labeled (test) dataset is assigned with a mask set to True
test_label_masks = np.ones(len(test_data), dtype=bool)
# val_label_masks = np.ones(len(val_examples), dtype=bool)
# val_dataloader=generate_data_loader(val_examples, val_label_masks, do_shuffle = False, balance_label_examples = False)

test_dataloader = generate_data_loader(test_data, test_label_masks, do_shuffle = False, balance_label_examples = False)

In [ ]:
for batch in train_dataloader:
    input_ids, attention_mask,label_mask, label  = batch
    print(input_ids.shape)
    print(input_ids)
    print(attention_mask)
    print(label)
    # print(label_mask)
    break

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_size=100, output_size=512, hidden_sizes=[512], dropout_rate=0.1):
        super(Generator, self).__init__()
        layers = []
        hidden_sizes = [noise_size] + hidden_sizes
        for i in range(len(hidden_sizes)-1):
            layers.extend([nn.Linear(hidden_sizes[i], hidden_sizes[i+1]), nn.LeakyReLU(0.2, inplace=True), nn.Dropout(dropout_rate)])

        layers.append(nn.Linear(hidden_sizes[-1],output_size))
        self.layers = nn.Sequential(*layers)

    def forward(self, noise):
        output_rep = self.layers(noise)
        return output_rep
class Discriminator(nn.Module):
    def __init__(self, input_size=512, hidden_sizes=[512], num_labels=2, dropout_rate=0.1):
        super(Discriminator, self).__init__()
        self.input_dropout = nn.Dropout(p=dropout_rate)
        layers = []
        hidden_sizes = [input_size] + hidden_sizes
        for i in range(len(hidden_sizes)-1):
            layers.extend([nn.Linear(hidden_sizes[i], hidden_sizes[i+1]), nn.LeakyReLU(0.2, inplace=True), nn.Dropout(dropout_rate)])

        self.layers = nn.Sequential(*layers) #per il flatten
        self.logit = nn.Linear(hidden_sizes[-1],num_labels+1) # +1 for the probability of this sample being fake/real.
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, input_rep):
        input_rep = self.input_dropout(input_rep)
        last_rep = self.layers(input_rep)
        logits = self.logit(last_rep)
        probs = self.softmax(logits)
        return last_rep, logits, probs

In [ ]:
# The config file is required to get the dimension of the vector produced by 
# the underlying transformer
config = AutoConfig.from_pretrained(model_name)
hidden_size = int(config.hidden_size)
# Define the number and width of hidden layers
hidden_levels_g = [256,512,768]
hidden_levels_d = [256,512,768]

#-------------------------------------------------
#   Instantiate the Generator and Discriminator
#-------------------------------------------------
generator = Generator(noise_size=noise_size, output_size=hidden_size, hidden_sizes=hidden_levels_g, dropout_rate=out_dropout_rate)
discriminator = Discriminator(input_size=hidden_size, hidden_sizes=hidden_levels_d, num_labels=2, dropout_rate=out_dropout_rate)

# Put everything in the GPU if available
if torch.cuda.is_available():    
  generator.cuda()
  discriminator.cuda()
  transformer.cuda()
  if multi_gpu:
    transformer = torch.nn.DataParallel(transformer)

# print(config)

In [ ]:
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

In [ ]:
training_stats = []

# Measure the total training time for the whole run.
total_t0 = time.time()

#models parameters
transformer_vars = [i for i in transformer.parameters()]
d_vars = transformer_vars + [v for v in discriminator.parameters()]
g_vars = [v for v in generator.parameters()]

#optimizer
dis_optimizer = torch.optim.AdamW(d_vars, lr=learning_rate_discriminator,weight_decay=1e-4)
gen_optimizer = torch.optim.AdamW(g_vars, lr=learning_rate_generator,weight_decay=1e-4) 

#scheduler
if apply_scheduler:
  num_train_examples = len(train_examples)
  num_train_steps = int(num_train_examples / batch_size * num_train_epochs)
  num_warmup_steps = int(num_train_steps * warmup_proportion)
  torch.optim.lr_scheduler.StepLR(dis_optimizer, step_size=2, gamma=0.5)
  torch.optim.lr_scheduler.StepLR(gen_optimizer, step_size=2, gamma=0.5)
  # scheduler_d = get_constant_schedule_with_warmup(dis_optimizer, 
                                           # num_warmup_steps = num_warmup_steps)
  # scheduler_g = get_constant_schedule_with_warmup(gen_optimizer, 
                                           # num_warmup_steps = num_warmup_steps)

# For each epoch...
for epoch_i in range(0, num_train_epochs):
    # ========================================
    #               Training
    # ========================================
    # Perform one full pass over the training set.
    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, num_train_epochs))
    print('Training...')

    # Measure how long the training epoch takes.
    t0 = time.time()

    # Reset the total loss for this epoch.
    tr_g_loss = 0
    tr_d_loss = 0

    # Put the model into training mode.
    transformer.train() 
    generator.train()
    discriminator.train()

    # For each batch of training data...
    for step, batch in enumerate(train_dataloader):

        # Progress update every print_each_n_step batches.
        if step % print_each_n_step == 0 and not step == 0:
            # Calculate elapsed time in minutes.
            elapsed = format_time(time.time() - t0)
            
            # Report progress.
            print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(train_dataloader), elapsed))

        # Unpack this training batch from our dataloader. 
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        b_label_mask = batch[3].to(device)

        real_batch_size = b_input_ids.shape[0]
     
        # Encode real data in the Transformer
        model_outputs = transformer(b_input_ids, attention_mask=b_input_mask)
        hidden_states = model_outputs[-1]
        # hidden_states = hidden_states.mean(dim=1)
        # Generate fake data that should have the same distribution of the ones
        # encoded by the transformer. 
        # First noisy input are used in input to the Generator
        noise = torch.zeros(real_batch_size, noise_size, device=device).uniform_(0, 1)
        # Gnerate Fake data
        gen_rep = generator(noise)

        # Generate the output of the Discriminator for real and fake data.
        # First, we put together the output of the tranformer and the generator
        # print(hidden_states.shape)
        # print(gen_rep.shape)
        disciminator_input = torch.cat([hidden_states, gen_rep], dim=0)
        # Then, we select the output of the disciminator
        features, logits, probs = discriminator(disciminator_input)

        # Finally, we separate the discriminator's output for the real and fake
        # data
        features_list = torch.split(features, real_batch_size)
        D_real_features = features_list[0]
        D_fake_features = features_list[1]
      
        logits_list = torch.split(logits, real_batch_size)
        D_real_logits = logits_list[0]
        D_fake_logits = logits_list[1]
        
        probs_list = torch.split(probs, real_batch_size)
        D_real_probs = probs_list[0]
        D_fake_probs = probs_list[1]

        #---------------------------------
        #  LOSS evaluation
        #---------------------------------
        # Generator's LOSS estimation
        g_loss_d = -1 * torch.mean(torch.log(1 - D_fake_probs[:,-1] + epsilon))
        g_feat_reg = torch.mean(torch.pow(torch.mean(D_real_features, dim=0) - torch.mean(D_fake_features, dim=0), 2))
        g_loss = g_loss_d + g_feat_reg
  
        # Disciminator's LOSS estimation
        logits = D_real_logits[:,0:-1]
        log_probs = F.log_softmax(logits, dim=-1)
        # The discriminator provides an output for labeled and unlabeled real data
        # so the loss evaluated for unlabeled data is ignored (masked)
        label2one_hot = torch.nn.functional.one_hot(b_labels, 2)
        # assert b_labels.min() >= -1 and b_labels.max() < 4, f"Invalid label found: {b_labels.min()}, {b_labels.max()}"
        per_example_loss = -torch.sum(label2one_hot * log_probs, dim=-1)
        per_example_loss = torch.masked_select(per_example_loss, b_label_mask.to(device).bool())
        labeled_example_count = per_example_loss.type(torch.float32).numel()

        # It may be the case that a batch does not contain labeled examples, 
        # so the "supervised loss" in this case is not evaluated
        if labeled_example_count == 0:
          D_L_Supervised = 0
        else:
          D_L_Supervised = torch.div(torch.sum(per_example_loss.to(device)), labeled_example_count)
                 
        D_L_unsupervised1U = -1 * torch.mean(torch.log(1 - D_real_probs[:, -1] + epsilon))
        D_L_unsupervised2U = -1 * torch.mean(torch.log(D_fake_probs[:, -1] + epsilon))
        d_loss = D_L_Supervised + D_L_unsupervised1U + D_L_unsupervised2U

        #---------------------------------
        #  OPTIMIZATION
        #---------------------------------
        # Avoid gradient accumulation
        gen_optimizer.zero_grad()
        dis_optimizer.zero_grad()

        # Calculate weigth updates
        # retain_graph=True is required since the underlying graph will be deleted after backward
        g_loss.backward(retain_graph=True)
        d_loss.backward() 
        
        # Apply modifications
        gen_optimizer.step()
        dis_optimizer.step()

        # A detail log of the individual losses
        #print("{0:.4f}\t{1:.4f}\t{2:.4f}\t{3:.4f}\t{4:.4f}".
        #      format(D_L_Supervised, D_L_unsupervised1U, D_L_unsupervised2U,
        #             g_loss_d, g_feat_reg))

        # Save the losses to print them later
        tr_g_loss += g_loss.item()
        tr_d_loss += d_loss.item()

        # Update the learning rate with the scheduler
        if apply_scheduler:
          scheduler_d.step()
          scheduler_g.step()

    # Calculate the average loss over all of the batches.
    avg_train_loss_g = tr_g_loss / len(train_dataloader)
    avg_train_loss_d = tr_d_loss / len(train_dataloader)             
    
    # Measure how long this epoch took.
    training_time = format_time(time.time() - t0)

    print("")
    print("  Average training loss generetor: {0:.3f}".format(avg_train_loss_g))
    print("  Average training loss discriminator: {0:.3f}".format(avg_train_loss_d))
    print("  Training epcoh took: {:}".format(training_time))
        
    # ========================================
    #     TEST ON THE EVALUATION DATASET
    # ========================================
    # After the completion of each training epoch, measure our performance on
    # our test set.
    print("")
    print("Running Test...")

    t0 = time.time()

    # Put the model in evaluation mode--the dropout layers behave differently
    # during evaluation.
    transformer.eval() #maybe redundant
    discriminator.eval()
    generator.eval()

    # Tracking variables 
    total_test_accuracy = 0
   
    total_test_loss = 0
    nb_test_steps = 0

    all_preds = []
    all_labels_ids = []

    #loss
    nll_loss = torch.nn.CrossEntropyLoss(ignore_index=-1)

    # Evaluate data for one epoch
    for batch in test_dataloader:
        
        # Unpack this training batch from our dataloader. 
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        
        # Tell pytorch not to bother with constructing the compute graph during
        # the forward pass, since this is only needed for backprop (training).
        with torch.no_grad():        
            model_outputs = transformer(b_input_ids, attention_mask=b_input_mask)
            hidden_states = model_outputs[-1]
            # hidden_states = hidden_states.mean(dim=1)
            _, logits, probs = discriminator(hidden_states)
            ###log_probs = F.log_softmax(probs[:,1:], dim=-1)
            filtered_logits = logits[:,0:-1]
            # Accumulate the test loss.
            total_test_loss += nll_loss(filtered_logits, b_labels)
            
        # Accumulate the predictions and the input labels
        _, preds = torch.max(filtered_logits, 1)
        all_preds += preds.detach().cpu()
        all_labels_ids += b_labels.detach().cpu()

    # Report the final accuracy for this validation run.
    all_preds = torch.stack(all_preds).numpy()
    all_labels_ids = torch.stack(all_labels_ids).numpy()
    test_accuracy = np.sum(all_preds == all_labels_ids) / len(all_preds)
    print("  Accuracy: {0:.3f}".format(test_accuracy))

    macro_f1 = f1_score(all_labels_ids, all_preds, average='macro')
    print("F1 Score: {:.3f}".format(macro_f1))

    # Calculate the average loss over all of the batches.
    avg_test_loss = total_test_loss / len(test_dataloader)
    avg_test_loss = avg_test_loss.item()
    
    # Measure how long the validation run took.
    test_time = format_time(time.time() - t0)
    
    print("  val Loss: {0:.3f}".format(avg_test_loss))
    print("  val took: {:}".format(test_time))

    # Record all statistics from this epoch.
    training_stats.append(
        {
            'epoch': epoch_i + 1,
            'Training Loss generator': avg_train_loss_g,
            'Training Loss discriminator': avg_train_loss_d,
            'Valid. Loss': avg_test_loss,
            'Valid. Accur.': test_accuracy,
            'F1 Score': macro_f1,
            'Training Time': training_time,
            'Test Time': test_time
        }
    )

In [ ]:
import matplotlib.pyplot as plt
# training_stats=training_stats[0:8]
# Extract accuracy and F1 score from training_stats
epochs = [entry['epoch'] for entry in training_stats]  # Get epochs
accuracy = [entry['Valid. Accur.'] for entry in training_stats]  # Get accuracy values
f1_scores = [entry['F1 Score'] for entry in training_stats]  # Get F1 scores (store in 'F1 Score' while logging)
test_loss=[entry['Valid. Loss'] for entry in training_stats]
d_loss=[entry['Training Loss discriminator'] for entry in training_stats]
g_loss=[entry['Training Loss generator'] for entry in training_stats]
# Plot accuracy and F1 score
plt.figure(figsize=(10, 6))
plt.plot(epochs, accuracy, label="Accuracy", marker="o")
plt.plot(epochs, f1_scores, label="F1 Score", marker="o")
# plt.plot(epochs,test_loss,label='tes_loss',marker="o")
# Adding titles and labels
plt.title("Accuracy and F1 Score over Epochs", fontsize=14)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Metrics", fontsize=12)
plt.xticks(epochs)
plt.legend(fontsize=12)
plt.grid(alpha=0.4)

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
plt.plot(epochs, test_loss, label="test_loss", marker="o")
plt.plot(epochs,d_loss , label="d_loss", marker="x")
plt.plot(epochs,g_loss , label="g_loss", marker="x")
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Metrics", fontsize=12)
plt.xticks(epochs)
plt.legend(fontsize=12)
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()